# Example queries: `calculated_column` (comstock_oedi)

Auto-generated from `tests/query_snapshots/calculated_column.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object


In [ ]:
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
)


## `calculated_column_elec_minus_gas`

Annual aggregation using a get_calculated_column-built enduse: electricity total minus natural gas total. Pins the operator-precedence and column-resolution path through get_calculated_column. The column_expr placeholders ($ELECTRICITY_TOTAL, $NATURAL_GAS_TOTAL) get resolved to per-schema column names by the specialized test function before construction.


In [ ]:
result = bsq.query(
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    enduses=[<sqlalchemy.sql.elements.Label at 0x11a577dd0; elec_minus_gas>],
)
result.head() if hasattr(result, 'head') else result


# shape: (14, 4)
comstock_building_type  sample_count  units_count  elec_minus_gas
 FullServiceRestaurant         13461  4556.431396    1.700981e+07
              Hospital            73   152.003100    5.765367e+08
            LargeHotel          2815  3317.336942    2.082084e+08
           LargeOffice          2271   849.927533    1.102256e+09
          MediumOffice          8708  2418.404703    8.987264e+08


## `calculated_column_ts_pivot_upgrade_pair`

TS calculated column on the pivot path: table='timeseries' + annual_only=False + upgrade_id != 0 + applied_only=False forces the upgrade-pair pivot subquery (NOT the upgrade_only short-circuit). The Label produced by get_calculated_column wraps an arithmetic expression of raw ts columns; the pivot must wrap that whole expression in a per-side CASE, not look up Label.name on ts.c. Without this entry, the pivot path's calculated-column handling is uncovered.


In [ ]:
result = bsq.query(
    annual_only=False,
    upgrade_id='1',
    applied_only=False,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
    enduses=[<sqlalchemy.sql.elements.Label at 0x11a5756d0; elec_minus_gas>],
)
result.head() if hasattr(result, 'head') else result


# shape: (12, 6)
state  timestamp  sample_count  units_count rows_per_sample  elec_minus_gas
   CO 2018-01-01        134649  84415.71204            2976    3.895435e+08
   CO 2018-02-01        134649  84415.71204            2688    3.323271e+08
   CO 2018-03-01        134649  84415.71204            2976    5.397073e+08
   CO 2018-04-01        134649  84415.71204            2880    5.935683e+08
   CO 2018-05-01        134649  84415.71204            2976    8.796155e+08


## `calculated_column_ts_pivot_with_baseline`

TS calculated column on the pivot path with include_baseline=True + applied_only=True. This combination forces the pivot subquery (upgrade_only short-circuit only fires when both include_baseline and include_savings are False) AND exercises the applied_in synthesis path that filters to applicable buildings. Together with the upgrade_pair entry above, these two pin both branches of the TS pivot's calculated-column handling.


In [ ]:
result = bsq.query(
    annual_only=False,
    upgrade_id='1',
    applied_only=True,
    include_baseline=True,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
    enduses=[<sqlalchemy.sql.elements.Label at 0x10d01bad0; elec_minus_gas>],
)
result.head() if hasattr(result, 'head') else result


# shape: (12, 7)
state  timestamp  sample_count  units_count rows_per_sample  elec_minus_gas__baseline  elec_minus_gas__upgrade
   CO 2018-01-01         78456 52826.632724            2976             -1.440088e+08             4.567994e+08
   CO 2018-02-01         78456 52826.632724            2688             -2.014714e+08             4.633219e+08
   CO 2018-03-01         78456 52826.632724            2976              9.043631e+07             3.328803e+08
   CO 2018-04-01         78456 52826.632724            2880              1.604109e+08             2.963054e+08
   CO 2018-05-01         78456 52826.632724            2976              3.359508e+08             2.977057e+08
